In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
np.random.seed(0)

In [3]:
threshold_min, threshold_max, threshold_delta = 0., 1., 0.1

In [4]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [5]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds
    thresholds_m = np.maximum(0, thresholds - (bayesian_update(priors) / c))

    if len(thresholds_m) > 1:
        priors_updated = []
        thresholds_updated = []
        thresholds_m_updated = []
        prev = -np.inf
        for i, threshold_m in enumerate(thresholds_m):
            if threshold_m <= prev:
                del priors_updated[i-1]
                del thresholds_updated[i-1]
                del thresholds_m_updated[i-1]

                priors_updated.append(priors[i-1] + priors[i])
            else:
                priors_updated.append(priors[i])
            thresholds_updated.append(thresholds[i])
            thresholds_m_updated.append(thresholds_m[i])
            prev = threshold_m
    else:
        priors_updated = priors
        thresholds_updated = thresholds
        thresholds_m_updated = thresholds_m

    return thresholds_m_updated, priors_updated, thresholds_updated

In [6]:
def balance_priors(priors, random=False):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [7]:
def accuracy_loss(thresholds, x_manipulation, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        if thresholds[i] < threshold_true:
            loss = (threshold_true - x_manipulation[i]) / (threshold_max - threshold_min)
        else:
            loss = np.abs(x_manipulation[i] - threshold_true) / (threshold_max - threshold_min)
        losses.append(loss)
    return np.array(losses)

In [8]:
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

priors = np.zeros_like(thresholds)
priors[1] = 0.78
priors[7] = 0.15

balance_priors(priors, random=True)
print(np.sum(priors))
# assert np.sum(priors) == 1

pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.0


,0,1,2,3,4,5,6,7,8
threshold,0.10,0.20,0.300,0.400,0.50,0.600,0.700,0.80,0.900
priors,0.01,0.78,0.013,0.011,0.01,0.008,0.012,0.15,0.008


In [13]:
c = 5
threshold_true = 0.5
n = len(thresholds)
partitions = [[i] for i in range(n)]
# partitions = [[0,1], [2,3], [4,5], [6,7,8]]

In [14]:
results = {
    "threshold": [],
    "prior": [],
    "partition": [],
    "accuracy_loss": [],
    "partition_loss": [],
    "manip_threshold": [],
}

for i, partition in enumerate(partitions):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]
    x_manipulation_p, priors_p, threshold_p = manipulation_thresholds(threshold_p, priors_p, c)
    acc_loss = accuracy_loss(threshold_p, x_manipulation_p, threshold_true)
    partition_loss = np.dot(acc_loss, bayesian_update(priors_p))
    
    for ti, t in enumerate(threshold_p):
        results["threshold"].append(t)
        results["prior"].append(priors_p[ti])
        results["partition"].append(f"{i}")
        results["accuracy_loss"].append(acc_loss[ti])
        results["partition_loss"].append(partition_loss)
        results["manip_threshold"].append(x_manipulation_p[ti])

In [15]:
df = pd.DataFrame(results).T
df

,0,1,2,3,4,5,6,7,8
threshold,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
prior,0.009803,0.78,0.012775,0.010767,0.009733,0.007568,0.011537,0.15,0.007816
partition,0,1,2,3,4,5,6,7,8
accuracy_loss,0.5,0.5,0.4,0.3,0.2,0.1,0.0,0.1,0.2
partition_loss,0.5,0.5,0.4,0.3,0.2,0.1,0.0,0.1,0.2
manip_threshold,0.0,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7


In [17]:
print(f"Accuracy Loss: {np.dot(df.T["accuracy_loss"].values, df.T["prior"].values):.4f}")

Accuracy Loss: 0.4225


In [21]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        # Option 1: put `first` in each existing block
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        # Option 2: put `first` in its own new block
        yield [[first]] + smaller

In [23]:
parts = set_partitions(list(range(5)))
partitions = []
for part in parts:
    partitions.append(part)
len(partitions)

52